# Notebook 02 - The SVI/SSVI Benchmark: Full Replication of Gatheral & Jacquier (2014) + Real SPX Data

**Purpose.** This notebook is the parametric backbone of the thesis. It (i) **replicates every technique** of *Arbitrage-free SVI volatility surfaces* (Gatheral & Jacquier, 2014), validating each against the paper's own numbers, and (ii) applies the calibration machinery to the **real SPX dataset** (`option_prices_clean.parquet` from NB01) to produce the quantified benchmark that the deep-learning models of NB03+ must beat.

**Paper coverage map:**

| # | Technique | Paper section |
|---|---|---|
| 1 | Raw / Natural / Jump-Wings parameterizations + exact conversions | 3.1–3.3, L3.1–3.2 |
| 2 | Static arbitrage: Durrleman $g(k)\ge0$, wing-slope condition, calendar crossedness | 2.1–2.2, D5.1 |
| 3 | Quasi-explicit calibration (Zeliade [27]) — our production calibrator | 5.2 (ref [27]) |
| 4 | The paper's own recipe: square-root-SVI init + crossedness penalty | 5.2 |
| 5 | Butterfly-elimination algorithm via SVI-JW (Example 5.1 reproduced) | 5.1 |
| 6 | SSVI variants: power-law, Heston-like, modified power-law (fully arbitrage-free) + Theorems 4.1/4.2 | 4 |
| 7 | Arbitrage-free interpolation & extrapolation | 5.3, L5.1, T4.3 |
| 8 | **Real-data benchmark**: SVI per slice (by `exdate`), SSVI per day, arbitrage audit | — |

> Note on Lemma 3.2: deriving the JW→raw inverse we correct a missing term in the paper's printed statement; our version is validated numerically (it recovers the raw Vogt parameters exactly).


In [ ]:
def eliminate_butterfly_jw(jw):
    vt, psit, pt = jw["vt"], jw["psit"], jw["pt"]
    ct_new = pt + 2 * psit
    vtil_new = vt * 4 * pt * ct_new / (pt + ct_new) ** 2
    out = dict(jw); out["ct"] = ct_new; out["vtil"] = vtil_new
    return out

jw_fixed = eliminate_butterfly_jw(raw_to_jw(**vogt, t=1.0))
print(f"c'_t  = {jw_fixed['ct']:.7f}   (paper: 0.3493158)")
print(f"vtil' = {jw_fixed['vtil']:.7f}   (paper: 0.0154818)")
raw_fixed = jw_to_raw(**jw_fixed, t=1.0)
mb, _ = svi_butterfly_ok(vogt); ma, _ = svi_butterfly_ok(raw_fixed)
print(f"min g: before = {mb:.4f} (arbitrage)  ->  after = {ma:.4f} (arbitrage-free)")

kk = np.linspace(-1.5, 1.5, 500)
d0, g0 = risk_neutral_density(kk, vogt)
d1, g1 = risk_neutral_density(kk, raw_fixed)
fig = make_subplots(rows=1, cols=3, subplot_titles=("Smile w(k)", "g(k)", "Implied density"))
for y0, y1, col in [(svi_raw(kk, **vogt), svi_raw(kk, **raw_fixed), 1), (g0, g1, 2), (d0, d1, 3)]:
    fig.add_trace(go.Scatter(x=kk, y=y0, name="Vogt (arbitrageable)",
                             line=dict(color="#ef553b"), showlegend=(col == 1)), 1, col)
    fig.add_trace(go.Scatter(x=kk, y=y1, name="fixed (butterfly-free)",
                             line=dict(color="#00cc96", dash="dash"), showlegend=(col == 1)), 1, col)
for col in (2, 3):
    fig.add_hline(y=0, line_dash="dot", row=1, col=col)
fig.update_layout(width=1100, height=380, title="§5.1 butterfly elimination — Example 5.1 reproduced")
fig.show()


c'_t  = 0.3493158   (paper: 0.3493158)
vtil' = 0.0154818   (paper: 0.0154818)
min g: before = -0.0329 (arbitrage)  ->  after = 0.2704 (arbitrage-free)


In [1]:
# --- config ---
REAL_PARQUET  = Path(os.environ.get("THESIS_OPT_PARQUET", "data/clean/option_prices_clean.parquet"))
OUT_DIR       = Path(os.environ.get("THESIS_OUT_DIR", "data/clean")); OUT_DIR.mkdir(parents=True, exist_ok=True)
MIN_PTS_SLICE = 6                   # min quotes per expiration for a 5-parameter SVI
MIN_SLICES    = 2                   # min slices per day for an SSVI fit
LIMIT_DATES   = 8                   # None = all days; int = quick pass
WEIGHT_MODE   = "inv_spread"        # "uniform" | "inv_spread" (papers weight by liquidity)
HOLDOUT_FRAC  = 0.20                # per-slice held-out quotes (generalization metric)
N_JOBS        = 4                   # joblib workers for the day loop (1 = sequential)
DOMAINS = {                         # run BOTH: full quoted domain and the papers-style restricted one
    "full":  dict(k_lo=None, k_hi=None, dte_min=None),
    "paper": dict(k_lo=-0.6, k_hi=0.4, dte_min=10),
}

PLOT_W, PLOT_H = 900, 420


## 1. The three SVI parameterizations and their exact conversions

**Raw** (3.1): $w(k)=a+b\big(\rho(k-m)+\sqrt{(k-m)^2+\sigma^2}\big)$ — tractable, the calibration workhorse.
**Natural** (3.2): $w(k)=\Delta+\tfrac{\omega}{2}\big(1+\zeta\rho(k-\mu)+\sqrt{(\zeta(k-\mu)+\rho)^2+1-\rho^2}\big)$ — the bridge to SSVI.
**Jump-Wings** (3.5): $(v_t,\psi_t,p_t,c_t,\tilde v_t)$ = ATM variance, ATM skew, put-wing slope, call-wing slope, minimum variance — trader-readable quantities and the vehicle of the butterfly-elimination algorithm.

Conversions: raw↔natural (Lemma 3.1) and raw↔JW (Lemma 3.2, corrected term).

In [2]:
# ---------- raw SVI + derivatives + Durrleman g ----------
def svi_raw(k, a, b, rho, m, sigma):
    k = np.asarray(k, float); return a + b * (rho * (k - m) + np.sqrt((k - m) ** 2 + sigma ** 2))
def svi_prime(k, a, b, rho, m, sigma):
    k = np.asarray(k, float); return b * (rho + (k - m) / np.sqrt((k - m) ** 2 + sigma ** 2))
def svi_double(k, a, b, rho, m, sigma):
    k = np.asarray(k, float); return b * sigma ** 2 / ((k - m) ** 2 + sigma ** 2) ** 1.5
def durrleman_g(k, w, wp, wpp):
    return (1 - k * wp / (2 * w)) ** 2 - (wp ** 2 / 4) * (1 / w + 0.25) + wpp / 2

# ---------- raw <-> natural (Lemma 3.1) ----------
def raw_to_natural(a, b, rho, m, sigma):
    r = np.sqrt(1 - rho ** 2)
    return dict(Delta=a - b * sigma / r * (1 - rho ** 2), mu=m + rho * sigma / r,
                rho=rho, omega=2 * b * sigma / r, zeta=r / sigma)

def natural_to_raw(Delta, mu, rho, omega, zeta):
    r = np.sqrt(1 - rho ** 2)
    b = 0.5 * omega * zeta
    sigma = r / zeta
    return dict(a=Delta + 0.5 * omega * (1 - rho ** 2), b=b, rho=rho,
                m=mu - rho / zeta, sigma=sigma)

# ---------- raw <-> Jump-Wings (Lemma 3.2, corrected inverse) ----------
def raw_to_jw(a, b, rho, m, sigma, t):
    w0 = a + b * (-rho * m + np.sqrt(m ** 2 + sigma ** 2))     # ATM total variance w(0)
    sq = np.sqrt(w0)
    return dict(
        vt=w0 / t,
        psit=b / (2 * sq) * (rho - m / np.sqrt(m ** 2 + sigma ** 2)),
        pt=b * (1 - rho) / sq,
        ct=b * (1 + rho) / sq,
        vtil=(a + b * sigma * np.sqrt(1 - rho ** 2)) / t,
    )

def jw_to_raw(vt, psit, pt, ct, vtil, t):
    wt = vt * t; sq = np.sqrt(wt)
    b = sq * (ct + pt) / 2
    rho = (ct - pt) / (ct + pt)
    beta = np.clip(rho - 2 * psit * sq / b, -0.9999999, 0.9999999)
    alpha = np.sign(beta) * np.sqrt(1 / beta ** 2 - 1)
    r = np.sqrt(1 - rho ** 2)
    # (vt - vtil) t = b m (1/beta - rho - alpha r)   [sqrt(m^2+s^2)=m/beta handles m<0]
    denom = b * (1.0 / beta - rho - alpha * r)
    m = (vt - vtil) * t / denom if abs(denom) > 1e-14 else 0.0
    sigma = alpha * m
    return dict(a=vtil * t - b * sigma * r, b=b, rho=rho, m=m, sigma=sigma)


In [3]:
# --- validation 1: random round-trips raw <-> natural <-> JW ---
rng = np.random.default_rng(0)
maxerr = 0.0
for _ in range(2000):
    p = dict(a=rng.uniform(0, .05), b=rng.uniform(.05, .3), rho=rng.uniform(-.9, .9),
             m=rng.uniform(-.3, .3), sigma=rng.uniform(.05, .5))
    t = rng.uniform(.1, 2)
    bn = natural_to_raw(**raw_to_natural(**p))
    bj = jw_to_raw(**raw_to_jw(t=t, **p), t=t)
    for key in p:
        maxerr = max(maxerr, abs(p[key] - bn[key]), abs(p[key] - bj[key]))
print(f"max round-trip error (raw<->natural<->JW, 2000 draws): {maxerr:.2e}")

# --- validation 2: Vogt example -> reproduce the paper's JW values ---
vogt = dict(a=-0.0410, b=0.1331, rho=0.3060, m=0.3586, sigma=0.4153)
jw = raw_to_jw(**vogt, t=1.0)
print("\nJW of Vogt   (paper: 0.01742625, -0.1752111, 0.6997381, 1.316798, 0.0116249)")
print("  computed :", {k: round(v, 7) for k, v in jw.items()})
print("  raw back :", {k: round(v, 4) for k, v in jw_to_raw(**jw, t=1.0).items()})


max round-trip error (raw<->natural<->JW, 2000 draws): 2.72e-10

JW of Vogt   (paper: 0.01742625, -0.1752111, 0.6997381, 1.316798, 0.0116249)
  computed : {'vt': 0.0174263, 'psit': -0.1752111, 'pt': 0.6997381, 'ct': 1.3167982, 'vtil': 0.0116249}
  raw back : {'a': -0.041, 'b': 0.1331, 'rho': 0.306, 'm': 0.3586, 'sigma': 0.4153}


## 2. Static arbitrage in practice

**Butterfly** (Lemma 2.2): a slice is butterfly-arbitrage-free iff (i) $g(k)\ge0$ for all $k$ **and** (ii) $\lim_{k\to\infty}d_+(k)=-\infty$ — for raw SVI, (ii) is the wing-slope condition $b(1+|\rho|)<2$ (Lee's bound).

**Calendar** (Def. 5.1): two slices must not cross; the *crossedness* measures how much the shorter slice exceeds the longer one.

We demonstrate both on the **Vogt smile** — a perfectly innocent-looking slice hiding a negative density.

In [4]:
def svi_butterfly_ok(params, k_lo=-2.0, k_hi=2.0, n=800):
    k = np.linspace(k_lo, k_hi, n)
    g = durrleman_g(k, svi_raw(k, **params), svi_prime(k, **params), svi_double(k, **params))
    wings_ok = (params["b"] * (1 + params["rho"]) < 2) and (params["b"] * (1 - params["rho"]) < 2)
    return float(g.min()), bool(wings_ok)

def crossedness(p_short, p_long, k_lo=-1.5, k_hi=1.5, n=400):
    k = np.linspace(k_lo, k_hi, n)
    return float(max(0.0, (svi_raw(k, **p_short) - svi_raw(k, **p_long)).max()))

def risk_neutral_density(k, params):
    w = svi_raw(k, **params); wp = svi_prime(k, **params); wpp = svi_double(k, **params)
    g = durrleman_g(k, w, wp, wpp)
    dm = -k / np.sqrt(w) - np.sqrt(w) / 2
    return g / np.sqrt(2 * np.pi * w) * np.exp(-0.5 * dm ** 2), g

ming, wok = svi_butterfly_ok(vogt)
print(f"Vogt: min g = {ming:.4f} ({'OK' if ming >= 0 else 'BUTTERFLY ARBITRAGE'}), wing slopes < 2: {wok}")

kk = np.linspace(-1.5, 1.5, 500)
dens, g = risk_neutral_density(kk, vogt)
fig = make_subplots(rows=1, cols=3, subplot_titles=(
    "Total variance w(k) — looks innocent", "g(k) dips below 0 → arbitrage", "Implied density goes NEGATIVE"))
fig.add_trace(go.Scatter(x=kk, y=svi_raw(kk, **vogt), name="w(k)"), 1, 1)
fig.add_trace(go.Scatter(x=kk, y=g, name="g(k)"), 1, 2)
fig.add_trace(go.Scatter(x=kk, y=np.where(g < 0, g, np.nan), mode="lines",
                         line=dict(color="red", width=4), name="g<0"), 1, 2)
fig.add_hline(y=0, line_dash="dot", row=1, col=2)
fig.add_trace(go.Scatter(x=kk, y=dens, name="density"), 1, 3)
fig.add_trace(go.Scatter(x=kk, y=np.where(dens < 0, dens, np.nan), mode="lines",
                         line=dict(color="red", width=4), name="p<0"), 1, 3)
fig.add_hline(y=0, line_dash="dot", row=1, col=3)
fig.update_layout(width=1100, height=380, showlegend=False,
                  title="The Vogt smile (paper Example 3.1): hidden butterfly arbitrage")
fig.show()


Vogt: min g = -0.0329 (BUTTERFLY ARBITRAGE), wing slopes < 2: True


## 3. Quasi-explicit SVI calibration (production calibrator)

The robust way to fit raw SVI (Zeliade white paper, ref [27] of Gatheral–Jacquier). Freezing $(m,\sigma)$ and letting $y=(k-m)/\sigma$ turns SVI into $w=a+dy+c\sqrt{y^2+1}$ — **linear** in $(a,d,c)$ with $c=b\sigma$, $d=\rho b\sigma$. The inner problem is a convex least squares under the linear wing/no-arbitrage constraints; only $(m,\sigma)$ are searched in 2-D (Nelder–Mead with restarts). No local-minima roulette.

In [5]:
def _svi_inner(m, s, k, w, wt):
    y = (k - m) / s; z = np.sqrt(y * y + 1.0)
    A = np.column_stack([np.ones_like(y), y, z])
    def obj(x):  r = A @ x - w; return float(np.sum(wt * r * r))
    def grad(x): r = A @ x - w; return 2.0 * A.T @ (wt * r)
    wmax = float(max(w.max(), 1e-6))
    cons = [
        {"type": "ineq", "fun": lambda x: x[2]},
        {"type": "ineq", "fun": lambda x: 4 * s - x[2]},
        {"type": "ineq", "fun": lambda x: x[2] - x[1]},
        {"type": "ineq", "fun": lambda x: x[2] + x[1]},
        {"type": "ineq", "fun": lambda x: (4 * s - x[2]) - x[1]},
        {"type": "ineq", "fun": lambda x: x[1] - (x[2] - 4 * s)},
        {"type": "ineq", "fun": lambda x: x[0]},
        {"type": "ineq", "fun": lambda x: wmax - x[0]},
    ]
    res = minimize(obj, np.array([np.median(w), 0.0, min(2 * s, wmax)]), jac=grad,
                   method="SLSQP", constraints=cons, options={"maxiter": 200, "ftol": 1e-14})
    return res.x, res.fun

def fit_svi_slice(k, w, weights=None, restarts=((0.0, 0.1), (0.0, 0.2), (-0.05, 0.3))):
    k = np.asarray(k, float); w = np.asarray(w, float)
    wt = np.ones_like(w) if weights is None else np.asarray(weights, float)
    best = None
    for (m0, s0) in restarts:
        r = minimize(lambda ms: _svi_inner(ms[0], np.exp(ms[1]), k, w, wt)[1],
                     [m0, np.log(s0)], method="Nelder-Mead",
                     options={"xatol": 1e-6, "fatol": 1e-14, "maxiter": 400})
        m, ls = r.x; s = float(np.exp(ls))
        x, f = _svi_inner(m, s, k, w, wt)
        if best is None or f < best[0]:
            a, d, c = x; b = c / s
            rho = float(np.clip(d / c, -0.9999, 0.9999)) if c > 1e-12 else 0.0
            best = (f, dict(a=float(a), b=float(b), rho=rho, m=float(m), sigma=s))
    p = best[1]
    return p, float(np.sqrt(np.mean((svi_raw(k, **p) - w) ** 2)))

def svi_min_g(params, k_lo, k_hi, n=400):
    k = np.linspace(k_lo, k_hi, n)
    return float(np.min(durrleman_g(k, svi_raw(k, **params),
                                    svi_prime(k, **params), svi_double(k, **params))))


In [6]:
# --- synthetic validation: recover a known SSVI slice under 1% noise ---
def ssvi_w(k, theta, rho, eta, gamma):
    k = np.asarray(k, float); phi = eta * theta ** (-gamma)
    return 0.5 * theta * (1 + rho * phi * k + np.sqrt((phi * k + rho) ** 2 + (1 - rho ** 2)))

TRUE = dict(rho=-0.5, eta=0.8, gamma=0.4)
theta0 = 0.04 * 0.5
k_obs = np.sort(rng.uniform(-0.5, 0.35, 14))
w_obs = ssvi_w(k_obs, theta0, **TRUE) * (1 + 0.01 * rng.standard_normal(len(k_obs)))
p_fit, rmse = fit_svi_slice(k_obs, w_obs)
print("fitted SVI:", {k: round(v, 4) for k, v in p_fit.items()})
print(f"RMSE(w) = {rmse:.2e} | min g = {svi_min_g(p_fit, -0.6, 0.45):.4f} (>=0: arbitrage-free)")

kk = np.linspace(-0.6, 0.45, 300)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Quasi-explicit SVI fit", "g(k) of the fit"))
fig.add_trace(go.Scatter(x=k_obs, y=w_obs, mode="markers", name="noisy market",
                         marker=dict(color="black", size=7)), 1, 1)
fig.add_trace(go.Scatter(x=kk, y=svi_raw(kk, **p_fit), name="calibrated SVI"), 1, 1)
fig.add_trace(go.Scatter(x=kk, y=ssvi_w(kk, theta0, **TRUE), name="true SSVI",
                         line=dict(dash="dash")), 1, 1)
gfit = durrleman_g(kk, svi_raw(kk, **p_fit), svi_prime(kk, **p_fit), svi_double(kk, **p_fit))
fig.add_trace(go.Scatter(x=kk, y=gfit, name="g(k)"), 1, 2)
fig.add_hline(y=0, line_dash="dot", row=1, col=2)
fig.update_layout(width=PLOT_W, height=PLOT_H, title="Calibrator validation on synthetic data")
fig.show()


fitted SVI: {'a': 0.0091, 'b': 0.0339, 'rho': -0.6661, 'm': 0.1078, 'sigma': 0.2279}
RMSE(w) = 2.75e-04 | min g = 0.4398 (>=0: arbitrage-free)


## 4. The paper's own calibration recipe (§5.2)

The alternative route Gatheral–Jacquier actually describe: (a) fit a **square-root SVI** surface — SSVI with $\gamma=\tfrac12$, whose JW slopes are maturity-independent — as the initial guess; (b) refine **slice-by-slice**, minimizing the fit error **plus a heavy crossedness penalty** against the neighboring slice, which eliminates calendar arbitrage. We demo it on a synthetic surface with an artificially injected calendar crossing.

In [7]:
def sqrt_svi_init(thetas, k_lists, w_lists):
    TH = np.asarray(thetas, float)
    def sse(p):
        rho, eta = p
        return sum(np.sum((ssvi_w(kk, th, rho, eta, 0.5) - wm) ** 2)
                   for th, kk, wm in zip(TH, k_lists, w_lists))
    best = None
    for init in [(-0.5, 1.0), (-0.7, 0.5), (-0.3, 1.5)]:
        r = minimize(sse, init, method="Nelder-Mead")
        if best is None or r.fun < best.fun: best = r
    rho, eta = best.x
    init_raw = [natural_to_raw(Delta=0.0, mu=0.0, rho=rho, omega=th, zeta=eta * th ** (-0.5))
                for th in TH]
    return init_raw, dict(rho=float(rho), eta=float(eta))

def calibrate_surface_paper(taus, k_lists, w_lists, lam=1e4):
    order = np.argsort(taus)
    taus = [taus[i] for i in order]
    k_lists = [k_lists[i] for i in order]; w_lists = [w_lists[i] for i in order]
    thetas0 = [float(np.interp(0.0, k_lists[i], w_lists[i])) for i in range(len(taus))]
    init_raw, _ = sqrt_svi_init(thetas0, k_lists, w_lists)
    fitted = []
    for i, (kk, wm) in enumerate(zip(k_lists, w_lists)):
        p0 = np.array([init_raw[i][c] for c in ("a", "b", "rho", "m", "sigma")])
        prev = fitted[i - 1] if i > 0 else None
        def obj(x):
            a, b, rho, m, s = x
            if b < 0 or s <= 1e-4 or abs(rho) >= 1: return 1e6
            p = dict(a=a, b=b, rho=rho, m=m, sigma=s)
            pen = lam * crossedness(prev, p) ** 2 if prev is not None else 0.0
            return np.sum((svi_raw(kk, **p) - wm) ** 2) + pen
        r = minimize(obj, p0, method="Nelder-Mead",
                     options={"maxiter": 2000, "xatol": 1e-8, "fatol": 1e-12})
        fitted.append(dict(zip(("a", "b", "rho", "m", "sigma"), r.x)))
    return taus, fitted


In [8]:
# --- demo with an injected calendar crossing ---
taus_d = [0.1, 0.3, 0.6, 1.0]
k_ld, w_ld = [], []
for i, t in enumerate(taus_d):
    th = 0.04 * t
    kk = np.sort(rng.uniform(-0.5, 0.35, 14))
    wm = ssvi_w(kk, th, **TRUE) * (1 + 0.01 * rng.standard_normal(len(kk)))
    if i == 1:
        wm = wm * 1.5      # inflate one short slice -> calendar crossing
    k_ld.append(kk); w_ld.append(wm)

_, fit_naive = calibrate_surface_paper(taus_d, k_ld, w_ld, lam=0.0)
_, fit_pen   = calibrate_surface_paper(taus_d, k_ld, w_ld, lam=1e4)
def total_cross(fs): return sum(crossedness(fs[i], fs[i+1]) for i in range(len(fs)-1))
print(f"total crossedness  WITHOUT penalty: {total_cross(fit_naive):.5f}")
print(f"total crossedness  WITH    penalty: {total_cross(fit_pen):.5f}   (should be ~0)")

kk = np.linspace(-0.6, 0.45, 300)
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "No penalty: slices CROSS (calendar arbitrage)", "Crossedness penalty: crossing removed"))
colors = ["#636efa", "#ef553b", "#00cc96", "#ab63fa"]
for fits, col in [(fit_naive, 1), (fit_pen, 2)]:
    for p, t, c in zip(fits, taus_d, colors):
        fig.add_trace(go.Scatter(x=kk, y=svi_raw(kk, **p), name=f"τ={t}", legendgroup=f"{t}",
                                 showlegend=(col == 1), line=dict(color=c)), 1, col)
fig.update_layout(width=1000, height=400, title="Paper recipe (§5.2): slice-by-slice with crossedness penalty")
fig.show()


total crossedness  WITHOUT penalty: 0.11324
total crossedness  WITH    penalty: 0.00000   (should be ~0)


## 5. Butterfly elimination via Jump-Wings (§5.1) — Example 5.1 reproduced

The paper's flagship construction: given $(v_t,\psi_t,p_t)$, choosing
$$c'_t = p_t + 2\psi_t,\qquad \tilde v'_t = v_t\,\frac{4\,p_t\,c'_t}{(p_t+c'_t)^2}$$
**guarantees** a butterfly-arbitrage-free smile. Applied to the Vogt smile, the paper reports $(c'_t,\tilde v'_t)=(0.3493158,\ 0.0154818)$ — we must reproduce these exactly.

In [9]:
def eliminate_butterfly_jw(jw):
    vt, psit, pt = jw["vt"], jw["psit"], jw["pt"]
    ct_new = pt + 2 * psit
    vtil_new = vt * 4 * pt * ct_new / (pt + ct_new) ** 2
    out = dict(jw); out["ct"] = ct_new; out["vtil"] = vtil_new
    return out

jw_fixed = eliminate_butterfly_jw(raw_to_jw(**vogt, t=1.0))
print(f"c'_t  = {jw_fixed['ct']:.7f}   (paper: 0.3493158)")
print(f"vtil' = {jw_fixed['vtil']:.7f}   (paper: 0.0154818)")
raw_fixed = jw_to_raw(**jw_fixed, t=1.0)
mb, _ = svi_butterfly_ok(vogt); ma, _ = svi_butterfly_ok(raw_fixed)
print(f"min g: before = {mb:.4f} (arbitrage)  ->  after = {ma:.4f} (arbitrage-free)")

kk = np.linspace(-1.5, 1.5, 500)
d0, g0 = risk_neutral_density(kk, vogt)
d1, g1 = risk_neutral_density(kk, raw_fixed)
fig = make_subplots(rows=1, cols=3, subplot_titles=("Smile w(k)", "g(k)", "Implied density"))
for y0, y1, col in [(svi_raw(kk, **vogt), svi_raw(kk, **raw_fixed), 1), (g0, g1, 2), (d0, d1, 3)]:
    fig.add_trace(go.Scatter(x=kk, y=y0, name="Vogt (arbitrageable)",
                             line=dict(color="#ef553b"), showlegend=(col == 1)), 1, col)
    fig.add_trace(go.Scatter(x=kk, y=y1, name="fixed (butterfly-free)",
                             line=dict(color="#00cc96", dash="dash"), showlegend=(col == 1)), 1, col)
for col in (2, 3):
    fig.add_hline(y=0, line_dash="dot", row=1, col=col)
fig.update_layout(width=1100, height=380, title="§5.1 butterfly elimination — Example 5.1 reproduced")
fig.show()


c'_t  = 0.3493158   (paper: 0.3493158)
vtil' = 0.0154818   (paper: 0.0154818)
min g: before = -0.0329 (arbitrage)  ->  after = 0.2704 (arbitrage-free)


## 6. SSVI variants and the no-arbitrage theorems

Three choices of $\varphi$ define SSVI surfaces:
- **power-law** (Ex. 4.2): $\varphi(\theta)=\eta\theta^{-\gamma}$, $0<\gamma<1$;
- **Heston-like** (Ex. 4.1): $\varphi(\theta)=\frac{1}{\lambda\theta}\big(1-\frac{1-e^{-\lambda\theta}}{\lambda\theta}\big)$;
- **modified power-law** (Eq. 4.5): $\varphi(\theta)=\frac{\eta}{\theta^\gamma(1+\theta)^{1-\gamma}}$ — **fully free of static arbitrage** whenever $\eta(1+|\rho|)\le 2$.

Checks: Theorem 4.2 (butterfly: $\theta\varphi(1+|\rho|)<4$ and $\theta\varphi^2(1+|\rho|)\le4$) and Theorem 4.1 (calendar, verified numerically: $w$ non-decreasing in $\theta$).

In [10]:
def phi_powerlaw(theta, eta, gamma): return eta * theta ** (-gamma)
def phi_heston(theta, lam):
    x = lam * theta
    return (1.0 / x) * (1 - (1 - np.exp(-x)) / x)
def phi_modified(theta, eta, gamma): return eta / (theta ** gamma * (1 + theta) ** (1 - gamma))

def ssvi_w_phi(k, theta, rho, phi_val):
    k = np.asarray(k, float)
    return 0.5 * theta * (1 + rho * phi_val * k + np.sqrt((phi_val * k + rho) ** 2 + (1 - rho ** 2)))

def ssvi_butterfly_thm42(thetas, rho, phi_fun, **kw):
    TH = np.asarray(thetas, float); ph = phi_fun(TH, **kw)
    c1 = float(np.max(TH * ph * (1 + abs(rho))))
    c2 = float(np.max(TH * ph ** 2 * (1 + abs(rho))))
    return (c1 < 4 and c2 <= 4), c1, c2

def ssvi_calendar_num(thetas, rho, phi_fun, k_lo=-1.0, k_hi=1.0, n=200, **kw):
    TH = np.sort(np.asarray(thetas, float)); kk = np.linspace(k_lo, k_hi, n)
    W = np.vstack([ssvi_w_phi(kk, th, rho, phi_fun(th, **kw)) for th in TH])
    return bool(np.all(np.diff(W, axis=0) >= -1e-10))

TH = np.array([0.01, 0.02, 0.04, 0.08, 0.12]); RHO = -0.5
rows = [
    ("power-law (η=0.8, γ=0.4)",  phi_powerlaw, dict(eta=0.8, gamma=0.4)),
    ("Heston-like (λ=1.0)",       phi_heston,   dict(lam=1.0)),
    (f"modified (η={2/(1+abs(RHO)):.3f}, γ=0.5) [guaranteed]", phi_modified,
     dict(eta=2.0 / (1 + abs(RHO)) - 1e-6, gamma=0.5)),
]
for name, fn, kw in rows:
    ok, c1, c2 = ssvi_butterfly_thm42(TH, RHO, fn, **kw)
    cal = ssvi_calendar_num(TH, RHO, fn, **kw)
    print(f"{name:<45} butterfly(T4.2): {ok} (c1={c1:.3f}, c2={c2:.3f})   calendar: {cal}")

# visual: the three phi's smiles at theta grid
kk = np.linspace(-0.6, 0.5, 250)
fig = make_subplots(rows=1, cols=3, subplot_titles=[r[0].split(" [")[0] for r in rows])
for j, (name, fn, kw) in enumerate(rows, start=1):
    for th in TH:
        fig.add_trace(go.Scatter(x=kk, y=ssvi_w_phi(kk, th, RHO, fn(th, **kw)),
                                 name=f"θ={th}", showlegend=(j == 1)), 1, j)
fig.update_layout(width=1100, height=380,
                  title="SSVI slices under three φ choices (no slice crossing = calendar OK)")
fig.show()


power-law (η=0.8, γ=0.4)                      butterfly(T4.2): True (c1=0.336, c2=0.628)   calendar: True
Heston-like (λ=1.0)                           butterfly(T4.2): True (c1=0.087, c2=0.042)   calendar: True
modified (η=1.333, γ=0.5) [guaranteed]        butterfly(T4.2): True (c1=0.655, c2=2.640)   calendar: True


## 7. Arbitrage-free interpolation (Lemma 5.1) & extrapolation (§5.3, Theorem 4.3)

**Interpolation** between two arbitrage-free slices $t_1<t<t_2$: interpolate normalized **prices** with $\sqrt\theta$-weights,
$$\frac{C_t}{K_t}=\alpha_t\frac{C_1}{K_1}+(1-\alpha_t)\frac{C_2}{K_2},\qquad
\alpha_t=\frac{\sqrt{\theta_{t_2}}-\sqrt{\theta_t}}{\sqrt{\theta_{t_2}}-\sqrt{\theta_{t_1}}},$$
then invert back to total variance. **Extrapolation** beyond the last slice: $w(k,\theta_t)=w(k,\theta_{t_n})+(\theta_t-\theta_{t_n})$, arbitrage-free by Theorem 4.3.

In [11]:
def bs_call_fwd(k, w):
    sq = np.sqrt(w)
    return norm.cdf(-k / sq + sq / 2) - np.exp(k) * norm.cdf(-k / sq - sq / 2)

def w_from_call(k, c, lo=1e-6, hi=25.0):
    f = lambda w: bs_call_fwd(k, w) - c
    if f(lo) * f(hi) > 0: return np.nan
    return brentq(f, lo, hi, xtol=1e-12)

def interpolate_slice(k, w1, w2, th1, th2, th_t):
    alpha = (np.sqrt(th2) - np.sqrt(th_t)) / (np.sqrt(th2) - np.sqrt(th1))
    ct = alpha * bs_call_fwd(k, w1) + (1 - alpha) * bs_call_fwd(k, w2)
    return np.array([w_from_call(kk, cc) for kk, cc in zip(np.atleast_1d(k), np.atleast_1d(ct))])

th1, th2, th_mid, th_far = 0.02, 0.06, 0.04, 0.10
kk = np.linspace(-0.4, 0.3, 80)
w1 = ssvi_w(kk, th1, **TRUE); w2 = ssvi_w(kk, th2, **TRUE)
w_int = interpolate_slice(kk, w1, w2, th1, th2, th_mid)
w_ext = w2 + (th_far - th2)

wp = np.gradient(w_int, kk); wpp = np.gradient(wp, kk)
g_int = durrleman_g(kk, w_int, wp, wpp)
print(f"interpolated slice: min g = {np.nanmin(g_int[2:-2]):.4f} (>= 0: butterfly-free)")
print(f"extrapolated slice sits above the last slice everywhere: {bool(np.all(w_ext >= w2))}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=kk, y=w1, name=f"slice t1 (θ={th1})"))
fig.add_trace(go.Scatter(x=kk, y=w_int, name=f"interpolated (θ={th_mid})",
                         line=dict(color="black", dash="dash", width=3)))
fig.add_trace(go.Scatter(x=kk, y=w2, name=f"slice t2 (θ={th2})"))
fig.add_trace(go.Scatter(x=kk, y=w_ext, name=f"extrapolated (θ={th_far})", line=dict(dash="dot")))
fig.update_layout(width=PLOT_W, height=PLOT_H, xaxis_title="log-moneyness k",
                  yaxis_title="total variance w",
                  title="Arbitrage-free interpolation (L5.1) and extrapolation (T4.3)")
fig.show()


interpolated slice: min g = 0.5151 (>= 0: butterfly-free)
extrapolated slice sits above the last slice everywhere: True


## 8. SSVI global calibration — with corrected ATM-variance handling

Two methodological fixes over a naive pipeline, both of which change reported numbers:

1. **$\theta$ estimated from the quotes**, not from the fitted SVI's extrapolation to $k=0$. Long-maturity slices often have few quotes near the money; extrapolating the slice SVI there produces erratic $\theta$'s. We interpolate the observed total variance at $k=0$ (or take the nearest quote when the slice does not straddle the money).
2. **Monotonicity is diagnosed in maturity order, never after sorting.** The ATM total variance must be non-decreasing in $\tau$; we (a) *report* the raw inversions (a data/estimation diagnostic worth its own figure), then (b) repair them by **isotonic regression (PAVA)** before the SSVI fit, so the surface model receives a valid term structure. The calendar check on the fitted SSVI is likewise performed in $\tau$ order.

In [12]:
def estimate_theta_from_quotes(k, w):
    """ATM total variance from the slice's quotes: interpolate at k=0 if the slice
    straddles the money, else take the quote closest to k=0."""
    if k.min() <= 0.0 <= k.max():
        return float(np.interp(0.0, k, w))
    return float(w[np.argmin(np.abs(k))])

def pava_increasing(y):
    """Isotonic regression (pool-adjacent-violators), non-decreasing."""
    vals = [float(v) for v in y]; wts = [1.0] * len(vals); idx = [[i] for i in range(len(vals))]
    changed = True
    while changed:
        changed = False
        j = 0
        while j < len(vals) - 1:
            if vals[j] > vals[j + 1] + 1e-15:
                tot = wts[j] + wts[j + 1]
                vals[j] = (vals[j] * wts[j] + vals[j + 1] * wts[j + 1]) / tot
                wts[j] = tot; idx[j] = idx[j] + idx[j + 1]
                del vals[j + 1]; del wts[j + 1]; del idx[j + 1]
                changed = True
            else:
                j += 1
    out = np.empty(len(y))
    for v, ix in zip(vals, idx):
        out[ix] = v
    return out

def theta_diagnostics(taus, thetas):
    """Inversions of the RAW theta term structure, in maturity order (no sorting of theta!)."""
    order = np.argsort(taus)
    th = np.asarray(thetas, float)[order]
    d = np.diff(th)
    return dict(n_inversions=int(np.sum(d < -1e-10)),
                worst_drop=float(d.min()) if len(d) else 0.0)

# --- Fix 1: symmetric, IV-targeted weights for BOTH calibrators -------------
IV_TARGET_LOSS = True   # delta method: dsigma = dw / (2 sqrt(w tau))

def effective_weights(w_mkt, tau, wt_liq):
    """Liquidity weights, rescaled so a least-squares in w approximates a
    least-squares in IV. Folding the scale into the weights keeps the
    Zeliade inner problem linear."""
    wt = np.asarray(wt_liq, float).copy()
    if IV_TARGET_LOSS:
        wt = wt / (4.0 * np.maximum(np.asarray(w_mkt, float), 1e-10) * tau)
    return wt / wt.mean()

def fit_ssvi(thetas, k_list, w_list, wt_list=None):
    """Weighted SSVI fit — SAME weights as the per-slice SVI (symmetric objective)."""
    THv = np.asarray(thetas, float)
    if wt_list is None:
        wt_list = [np.ones_like(np.asarray(k, float)) for k in k_list]
    def sse(p):
        rho, eta, gamma = p
        tot = 0.0
        for th, kk, wm, wt in zip(THv, k_list, w_list, wt_list):
            r = ssvi_w(kk, th, rho, eta, gamma) - wm
            tot += float(np.sum(wt * r * r))
        return tot
    def phi(th, eta, gamma): return eta * th ** (-gamma)
    cons = [
        {"type": "ineq", "fun": lambda p: 0.999 - abs(p[0])},
        {"type": "ineq", "fun": lambda p: p[1] - 1e-6},
        {"type": "ineq", "fun": lambda p: p[2] - 1e-6},
        {"type": "ineq", "fun": lambda p: 0.999 - p[2]},   # Fix: gamma in (0,1), not (0,0.5]
        {"type": "ineq", "fun": lambda p: 4 - np.max(THv * phi(THv, p[1], p[2]) * (1 + abs(p[0])))},
        {"type": "ineq", "fun": lambda p: 4 - np.max(THv * phi(THv, p[1], p[2]) ** 2 * (1 + abs(p[0])))},
    ]
    best = None
    for init in [(-0.5, 1.0, 0.3), (-0.7, 0.5, 0.4), (-0.3, 1.5, 0.25)]:
        r = minimize(sse, init, method="SLSQP", constraints=cons,
                     options={"maxiter": 300, "ftol": 1e-14})
        if best is None or r.fun < best.fun: best = r
    rho, eta, gamma = best.x
    npts = sum(len(x) for x in k_list)
    return dict(rho=float(rho), eta=float(eta), gamma=float(gamma)), float(np.sqrt(best.fun / max(npts, 1)))

def ssvi_calendar_ok(taus, thetas, params, k_lo=-0.5, k_hi=0.5, n=200):
    """Calendar check in MATURITY order (the previous sorted-theta version silently
    masked term-structure inversions)."""
    order = np.argsort(taus)
    TH = np.asarray(thetas, float)[order]
    kk = np.linspace(k_lo, k_hi, n)
    W = np.vstack([ssvi_w(kk, th, **params) for th in TH])
    return bool(np.all(np.diff(W, axis=0) >= -1e-10))

# synthetic validation (unchanged machinery, corrected checks)
taus_s = np.array([0.1, 0.25, 0.5, 1.0, 1.5, 2.0])
rng = np.random.default_rng(0)
kl, wl, ths = [], [], []
for th_true in 0.04 * taus_s:
    kk = np.sort(rng.uniform(-0.6, 0.4, 16))
    wm = ssvi_w(kk, th_true, **TRUE) * (1 + 0.01 * rng.standard_normal(len(kk)))
    kl.append(kk); wl.append(wm)
    ths.append(estimate_theta_from_quotes(kk, wm))
diag = theta_diagnostics(taus_s, ths)
ths_iso = pava_increasing(np.asarray(ths)[np.argsort(taus_s)])
ssvi_p, ssvi_rmse = fit_ssvi(ths_iso, [kl[i] for i in np.argsort(taus_s)],
                             [wl[i] for i in np.argsort(taus_s)])
print("true SSVI  :", TRUE)
print("fitted SSVI:", {k: round(v, 4) for k, v in ssvi_p.items()},
      f"| RMSE={ssvi_rmse:.2e} | raw-theta inversions: {diag['n_inversions']} "
      f"| calendar (tau order): {'OK' if ssvi_calendar_ok(np.sort(taus_s), ths_iso, ssvi_p) else 'VIOLATION'}")


true SSVI  : {'rho': -0.5, 'eta': 0.8, 'gamma': 0.4}
fitted SSVI: {'rho': -0.5079, 'eta': 0.755, 'gamma': 0.412} | RMSE=5.44e-04 | raw-theta inversions: 0 | calendar (tau order): OK


## 9. Real-data benchmark v2 — hold-out, liquidity weights, cross-slice audit, two domains

Upgrades over the first driver, each fixing a bias identified in the first run:

- **Per-slice hold-out (20%)**: SVI/SSVI are fitted on 80% of each slice's quotes and *also* scored on the withheld 20% — the generalization metric that makes the NB03/NB04 comparison fair (they report hold-out; a benchmark scored only in-sample would be flattered).
- **Liquidity weighting** (`inv_spread`): wide-spread wing quotes no longer dominate the fit — the papers' choice.
- **Cross-slice crossedness**: butterfly was audited per slice, but calendar arbitrage lives *between* slices; we now measure the total crossedness of adjacent fitted SVI slices per day.
- **Raw-$\theta$ inversion count** per day (the diagnostic the corrected check exposes).
- **Two domains, run side by side**: the **full** quoted domain (honest about real difficulty — the bucket table showed 76% butterfly violations concentrated in 7–14d slices) and the **paper** domain ($k\in[-0.6,0.4]$, DTE ≥ 10) for direct comparability with the literature. Reporting both is the thesis-grade choice.
- **Parallel day loop** (`joblib`, `N_JOBS`).

In [13]:
import polars as pl
from joblib import Parallel, delayed

def slices_for_date(df_day, dom, rng_seed=0):
    """One slice per exdate, with per-slice train/holdout split and liquidity weights."""
    r = np.random.default_rng(rng_seed)
    out = []
    for _, sub in df_day.group_by("exdate"):
        sub = sub.sort("k")
        k = sub["k"].to_numpy(); iv = sub["iv_om"].to_numpy(); tau = float(sub["tau"][0])
        sp = sub["spread"].to_numpy() if "spread" in sub.columns else np.ones_like(k)
        ok = np.isfinite(k) & np.isfinite(iv) & (iv > 0)
        if dom["k_lo"] is not None:
            ok &= (k >= dom["k_lo"]) & (k <= dom["k_hi"])
        if dom["dte_min"] is not None:
            if tau * 365 < dom["dte_min"]:
                continue
        k, iv, sp = k[ok], iv[ok], sp[ok]
        if len(k) < MIN_PTS_SLICE + 2:      # need room for a holdout
            continue
        w = iv ** 2 * tau
        wt = (1.0 / np.maximum(sp, 1e-6)) if WEIGHT_MODE == "inv_spread" else np.ones_like(k)
        wt = wt / wt.mean()
        hold = r.random(len(k)) < HOLDOUT_FRAC
        if (~hold).sum() < MIN_PTS_SLICE:
            hold[:] = False
        out.append(dict(tau=tau, k=k, w=w, wt=wt, hold=hold))
    return sorted(out, key=lambda s: s["tau"])

def iv_rmse(k, w_hat, w_true, tau):
    return float(np.sqrt(np.mean((np.sqrt(np.maximum(w_hat, 1e-12) / tau)
                                  - np.sqrt(w_true / tau)) ** 2)))

# --- Fix 3a: butterfly repair on violating real slices ----------------------
def repair_butterfly(p, t):
    """Paper §5.1: move (c_t, vtil_t) to the guaranteed butterfly-free point."""
    return jw_to_raw(**eliminate_butterfly_jw(raw_to_jw(**p, t=t)), t=t)

# --- Fix 3b: calendar repair = paper §5.2 recipe on the real fits -----------
def repair_calendar(params_list, slice_data, lam=1e4, k_band=(-0.6, 0.6)):
    """Sequential weighted refit with a crossedness penalty against the previous
    (already repaired) slice. slice_data = list of (k, w, wt) training arrays."""
    fixed = []
    for i, (p0, (kk, wm, wt)) in enumerate(zip(params_list, slice_data)):
        prev = fixed[i - 1] if i > 0 else None
        x0 = np.array([p0[c] for c in ("a", "b", "rho", "m", "sigma")])
        def obj(x):
            a, b, rho, m, sg = x
            if b < 0 or sg <= 1e-4 or abs(rho) >= 1: return 1e6
            p = dict(a=a, b=b, rho=rho, m=m, sigma=sg)
            pen = lam * crossedness(prev, p, k_lo=k_band[0], k_hi=k_band[1]) ** 2 if prev is not None else 0.0
            return float(np.sum(wt * (svi_raw(kk, **p) - wm) ** 2)) + pen
        r = minimize(obj, x0, method="Nelder-Mead",
                     options={"maxiter": 1500, "xatol": 1e-8, "fatol": 1e-13})
        fixed.append(dict(zip(("a", "b", "rho", "m", "sigma"), r.x)))
    return fixed

def calibrate_day_v2(df_day, dom, rng_seed=0):
    slices = slices_for_date(df_day, dom, rng_seed)
    if len(slices) < MIN_SLICES:
        return None
    rows, params_list, taus, thetas = [], [], [], []
    k_tr_all, w_tr_all = [], []
    for s in slices:
        tr = ~s["hold"]; ho = s["hold"]
        wt_eff = effective_weights(s["w"][tr], s["tau"], s["wt"][tr])
        p, _ = fit_svi_slice(s["k"][tr], s["w"][tr], weights=wt_eff)
        ming = svi_min_g(p, s["k"].min() - 0.05, s["k"].max() + 0.05)
        # locate the violation (moneyness bucket of arbitrage)
        kk_g = np.linspace(s["k"].min() - 0.05, s["k"].max() + 0.05, 200)
        gg = durrleman_g(kk_g, svi_raw(kk_g, **p), svi_prime(kk_g, **p), svi_double(kk_g, **p))
        k_at_min_g = float(kk_g[np.argmin(gg)])
        rec = dict(tau=s["tau"], n=int(tr.sum()),
                   svi_rmse_iv=iv_rmse(s["k"][tr], svi_raw(s["k"][tr], **p), s["w"][tr], s["tau"]),
                   svi_rmse_iv_holdout=(iv_rmse(s["k"][ho], svi_raw(s["k"][ho], **p), s["w"][ho], s["tau"])
                                        if ho.sum() >= 2 else None),
                   svi_min_g=ming, k_at_min_g=k_at_min_g, **p)
        rows.append(rec)
        params_list.append(p); taus.append(s["tau"])
        thetas.append(estimate_theta_from_quotes(s["k"][tr], s["w"][tr]))
        k_tr_all.append(s["k"][tr]); w_tr_all.append(s["w"][tr])
        wt_tr_all.append(wt_eff)
    # cross-slice calendar audit on the fitted SVIs (adjacent maturities)
    cross = sum(crossedness(params_list[i], params_list[i + 1],
                            k_lo=max(-0.6, min(k.min() for k in k_tr_all)),
                            k_hi=min(0.6, max(k.max() for k in k_tr_all)))
                for i in range(len(params_list) - 1))
    tdiag = theta_diagnostics(np.array(taus), np.array(thetas))
    # SSVI on isotonized thetas (tau order)
    order = np.argsort(taus)
    ths_iso = pava_increasing(np.asarray(thetas)[order])
    #ssvi_p, _ = fit_ssvi(ths_iso, [k_tr_all[i] for i in order], [w_tr_all[i] for i in order])
    ssvi_p, _ = fit_ssvi(ths_iso, [k_tr_all[i] for i in order],
                         [w_tr_all[i] for i in order],
                         wt_list=[wt_tr_all[i] for i in order])
    cal_ok = ssvi_calendar_ok(np.asarray(taus)[order], ths_iso, ssvi_p)
    # butterfly repair, per violating slice
    for rec, p, s in zip(rows, params_list, slices):
        if rec["svi_min_g"] < -1e-6:
            tr = ~s["hold"]
            p_fix = repair_butterfly(p, s["tau"])
            rec["svi_rmse_iv_bflyfix"] = iv_rmse(s["k"][tr], svi_raw(s["k"][tr], **p_fix),
                                                 s["w"][tr], s["tau"])
            rec["svi_min_g_bflyfix"] = svi_min_g(p_fix, s["k"].min() - 0.05, s["k"].max() + 0.05)
        else:
            rec["svi_rmse_iv_bflyfix"] = None
            rec["svi_min_g_bflyfix"] = None

    # calendar repair when material crossedness is detected
    CROSS_TOL = 1e-4
    if cross > CROSS_TOL:
        sd = [(s["k"][~s["hold"]], s["w"][~s["hold"]], wt) 
              for s, wt in zip(slices, wt_tr_all)]
        params_cal = repair_calendar(params_list, sd)
        cross_after = sum(crossedness(params_cal[i], params_cal[i + 1])
                          for i in range(len(params_cal) - 1))
        rmse_cal = [iv_rmse(s["k"][~s["hold"]], svi_raw(s["k"][~s["hold"]], **p),
                            s["w"][~s["hold"]], s["tau"]) for s, p in zip(slices, params_cal)]
        for rec, r_ in zip(rows, rmse_cal):
            rec["svi_rmse_iv_calfix"] = r_
    else:
        cross_after = cross
        for rec in rows:
            rec["svi_rmse_iv_calfix"] = None
    # add "crossedness_after": float(cross_after) to the returned dict
    # SSVI in/holdout errors
    e_in, e_out = [], []
    for s, th in zip([slices[i] for i in order], ths_iso):
        tr = ~s["hold"]; ho = s["hold"]
        e_in.append(iv_rmse(s["k"][tr], ssvi_w(s["k"][tr], th, **ssvi_p), s["w"][tr], s["tau"]))
        if ho.sum() >= 2:
            e_out.append(iv_rmse(s["k"][ho], ssvi_w(s["k"][ho], th, **ssvi_p), s["w"][ho], s["tau"]))
    return dict(rows=rows, ssvi=ssvi_p,
                ssvi_rmse_iv=float(np.sqrt(np.mean(np.array(e_in) ** 2))),
                ssvi_rmse_iv_holdout=(float(np.sqrt(np.mean(np.array(e_out) ** 2))) if e_out else None),
                calendar_ok=cal_ok, crossedness=float(cross),
                theta_inversions=tdiag["n_inversions"], n_slices=len(slices))

import zlib
def stable_seed(d):
    return zlib.crc32(str(d).encode("utf-8"))

def run_domain(df, dates, dom, tag):
    def one(d):
        #return d, calibrate_day_v2(df.filter(pl.col("date") == d), dom, rng_seed=hash(str(d)) % 2**32)
        return d, calibrate_day_v2(df.filter(pl.col("date") == d), dom, rng_seed=stable_seed(d))
    results = Parallel(n_jobs=N_JOBS, prefer="threads")(delayed(one)(d) for d in dates)
    per_slice, per_day = [], []
    for d, res in results:
        if res is None: continue
        for r in res["rows"]:
            per_slice.append({"date": d, **r})
        per_day.append({"date": d, "n_slices": res["n_slices"],
                        "ssvi_rmse_iv": res["ssvi_rmse_iv"],
                        "ssvi_rmse_iv_holdout": res["ssvi_rmse_iv_holdout"],
                        "calendar_ok": res["calendar_ok"], "crossedness": res["crossedness"],
                        "theta_inversions": res["theta_inversions"],
                        **{f"ssvi_{k}": v for k, v in res["ssvi"].items()}})
    print(f"[{tag}] done: {len(per_slice)} slices, {len(per_day)} days")
    return pl.DataFrame(per_slice), pl.DataFrame(per_day)

if not REAL_PARQUET.exists():
    print(f"[info] {REAL_PARQUET} not found - real-data sections skipped. Run NB01 first.")
    bench = None
else:
    need = ["date", "exdate", "tau", "k", "iv_om", "is_otm"]
    lf = pl.scan_parquet(REAL_PARQUET).filter(pl.col("is_otm"))
    cols = lf.collect_schema().names()
    if "spread" in cols:
        need = need + ["spread"]
    df = lf.select(need).drop_nulls(["date", "exdate", "tau", "k", "iv_om"]).collect(engine="streaming")
    dates = df["date"].unique().sort().to_list()
    if LIMIT_DATES:
        dates = dates[:LIMIT_DATES]
    print(f"calibrating {len(dates)} day(s) on {len(DOMAINS)} domains...")
    bench = {name: run_domain(df, dates, dom, name) for name, dom in DOMAINS.items()}


calibrating 8 day(s) on 2 domains...
[full] done: 258 slices, 8 days
[paper] done: 246 slices, 8 days


### 9b. Aggregated metrics — full vs paper domain, in-sample vs hold-out

In [14]:
def summarize(slice_df, day_df, tag):
    svi_in  = slice_df["svi_rmse_iv"].to_numpy() * 100
    svi_out = slice_df["svi_rmse_iv_holdout"].drop_nulls().to_numpy() * 100
    ssvi_in = day_df["ssvi_rmse_iv"].to_numpy() * 100
    ssvi_out = day_df["ssvi_rmse_iv_holdout"].drop_nulls().to_numpy() * 100
    viol = (slice_df["svi_min_g"].to_numpy() < -1e-6).mean() * 100
    print(f"--- {tag} domain ---")
    print(f"SVI  in-sample {np.median(svi_in):.3f} | hold-out {np.median(svi_out):.3f} vol pts")
    print(f"SSVI in-sample {np.median(ssvi_in):.3f} | hold-out {np.median(ssvi_out):.3f} vol pts")
    print(f"butterfly-violating SVI slices: {viol:.2f}% | days with raw-theta inversions: "
          f"{(day_df['theta_inversions'] > 0).mean()*100:.0f}% | "
          f"median cross-slice crossedness: {float(day_df['crossedness'].median()):.2e}")
    return dict(tag=tag, svi_in=float(np.median(svi_in)), svi_out=float(np.median(svi_out)),
                ssvi_in=float(np.median(ssvi_in)), ssvi_out=float(np.median(ssvi_out)), viol=viol)

if bench is not None:
    summaries = [summarize(*bench[name], name) for name in bench]


--- full domain ---
SVI  in-sample 0.949 | hold-out 0.968 vol pts
SSVI in-sample 2.789 | hold-out 2.790 vol pts
butterfly-violating SVI slices: 6.20% | days with raw-theta inversions: 12% | median cross-slice crossedness: 3.31e-02
--- paper domain ---
SVI  in-sample 0.806 | hold-out 0.806 vol pts
SSVI in-sample 2.551 | hold-out 2.670 vol pts
butterfly-violating SVI slices: 2.44% | days with raw-theta inversions: 12% | median cross-slice crossedness: 1.05e-02


In [15]:
# --- maturity-bucket analysis (the table that localized the violations) ---
if bench is not None:
    slice_full, _ = bench["full"]
    sl = slice_full.with_columns([
        pl.when(pl.col("tau") * 365 <= 14).then(pl.lit("07-14d"))
          .when(pl.col("tau") * 365 <= 60).then(pl.lit("15-60d"))
          .when(pl.col("tau") * 365 <= 180).then(pl.lit("61-180d"))
          .otherwise(pl.lit(">180d")).alias("bucket"),
        pl.when(pl.col("k_at_min_g") < -0.6).then(pl.lit("deep put wing"))
          .when(pl.col("k_at_min_g") > 0.4).then(pl.lit("deep call wing"))
          .otherwise(pl.lit("core")).alias("viol_zone"),
    ])
    tab = (sl.group_by("bucket")
             .agg((pl.col("svi_rmse_iv").median() * 100).alias("rmse_med"),
                  ((pl.col("svi_min_g") < -1e-6).mean() * 100).alias("viol_pct"),
                  pl.len().alias("n"))
             .sort("bucket"))
    print(tab)
    zone = (sl.filter(pl.col("svi_min_g") < -1e-6).group_by("viol_zone")
              .agg(pl.len().alias("n_violations")).sort("n_violations", descending=True))
    print("\nWhere do butterfly violations sit (location of min g)?"); print(zone)

    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        "IV RMSE by maturity bucket (full domain)", "Butterfly-violation rate by bucket"))
    order = ["07-14d", "15-60d", "61-180d", ">180d"]
    tabp = tab.to_pandas().set_index("bucket").reindex(order)
    fig.add_trace(go.Bar(x=order, y=tabp["rmse_med"], marker_color="#636efa", showlegend=False), 1, 1)
    fig.add_trace(go.Bar(x=order, y=tabp["viol_pct"], marker_color="#ef553b", showlegend=False), 1, 2)
    fig.update_yaxes(title_text="vol pts", row=1, col=1)
    fig.update_yaxes(title_text="% slices with min g < 0", row=1, col=2)
    fig.update_layout(width=950, height=380,
                      title="Difficulty is localized: ultra-short maturities carry the arbitrage risk")
    fig.show()


shape: (4, 4)
┌─────────┬──────────┬───────────┬─────┐
│ bucket  ┆ rmse_med ┆ viol_pct  ┆ n   │
│ ---     ┆ ---      ┆ ---       ┆ --- │
│ str     ┆ f64      ┆ f64       ┆ u32 │
╞═════════╪══════════╪═══════════╪═════╡
│ 07-14d  ┆ 0.995194 ┆ 55.172414 ┆ 29  │
│ 15-60d  ┆ 1.003185 ┆ 0.0       ┆ 109 │
│ 61-180d ┆ 0.880514 ┆ 0.0       ┆ 56  │
│ >180d   ┆ 0.134515 ┆ 0.0       ┆ 64  │
└─────────┴──────────┴───────────┴─────┘

Where do butterfly violations sit (location of min g)?
shape: (1, 2)
┌───────────┬──────────────┐
│ viol_zone ┆ n_violations │
│ ---       ┆ ---          │
│ str       ┆ u32          │
╞═══════════╪══════════════╡
│ core      ┆ 16           │
└───────────┴──────────────┘


### 9c. Full-vs-paper domain comparison, and in-sample vs hold-out

In [16]:
if bench is not None:
    cats = ["SVI in", "SVI hold-out", "SSVI in", "SSVI hold-out"]
    fig = go.Figure()
    for s, color in zip(summaries, ["#636efa", "#00cc96"]):
        fig.add_trace(go.Bar(name=f"{s['tag']} domain", x=cats,
                             y=[s["svi_in"], s["svi_out"], s["ssvi_in"], s["ssvi_out"]],
                             marker_color=color))
    fig.update_layout(barmode="group", width=850, height=420,
                      yaxis_title="median IV RMSE (vol pts)",
                      title="Benchmark under both evaluation domains — the thesis reports both")
    fig.show()


In [ ]:
## 9d. Real-data smile gallery — the paper's Figure 4 analogue
def plot_day_smiles(df, slice_df, day_df, date, max_cols=4, max_slices=12):
    sl = slice_df.filter(pl.col("date") == date).sort("tau").head(max_slices)
    dd = day_df.filter(pl.col("date") == date)
    ssvi_p = dict(rho=float(dd["ssvi_rho"][0]), eta=float(dd["ssvi_eta"][0]),
                  gamma=float(dd["ssvi_gamma"][0]))
    day = df.filter(pl.col("date") == date)
    quotes = []
    for rec in sl.iter_rows(named=True):
        sub = day.filter((pl.col("tau") - rec["tau"]).abs() < 1e-9).sort("k")
        k = sub["k"].to_numpy(); iv = sub["iv_om"].to_numpy()
        quotes.append((rec, k, iv ** 2 * rec["tau"]))
    ths = pava_increasing([rec["theta_raw"] for rec, _, _ in quotes])
    n = len(quotes); ncols = min(max_cols, n); nrows = int(np.ceil(n / ncols))
    fig = make_subplots(rows=nrows, cols=ncols,
                        subplot_titles=[f"τ = {q[0]['tau']:.3f}y" for q in quotes])
    for i, ((rec, k, w), th) in enumerate(zip(quotes, ths)):
        rr, cc = divmod(i, ncols)
        kk = np.linspace(k.min(), k.max(), 150)
        p = {c: rec[c] for c in ("a", "b", "rho", "m", "sigma")}
        fig.add_trace(go.Scatter(x=k, y=np.sqrt(w / rec["tau"]), mode="markers",
                                 marker=dict(size=4, color="black"), name="market",
                                 showlegend=(i == 0)), rr + 1, cc + 1)
        fig.add_trace(go.Scatter(x=kk, y=np.sqrt(np.maximum(svi_raw(kk, **p), 1e-12) / rec["tau"]),
                                 line=dict(color="#ef553b"), name="SVI",
                                 showlegend=(i == 0)), rr + 1, cc + 1)
        fig.add_trace(go.Scatter(x=kk, y=np.sqrt(np.maximum(ssvi_w(kk, th, **ssvi_p), 1e-12) / rec["tau"]),
                                 line=dict(color="#00cc96", dash="dash"), name="SSVI",
                                 showlegend=(i == 0)), rr + 1, cc + 1)
    fig.update_layout(width=1150, height=270 * nrows,
                      title=f"{date} — market vs SVI vs SSVI (analogue of paper Fig. 4)")
    fig.show()

if bench is not None:
    slice_full, day_full = bench["full"]
    plot_day_smiles(df, slice_full, day_full, slice_full["date"].unique().sort()[0])

In [ ]:
def plot_day_total_variance(slice_df, date, k_band=(-0.6, 0.6)):
    sl = slice_df.filter(pl.col("date") == date).sort("tau")
    kk = np.linspace(*k_band, 240)
    fig = go.Figure()
    for rec in sl.iter_rows(named=True):
        p = {c: rec[c] for c in ("a", "b", "rho", "m", "sigma")}
        fig.add_trace(go.Scatter(x=kk, y=svi_raw(kk, **p), name=f"τ={rec['tau']:.3f}"))
    fig.update_layout(width=PLOT_W, height=460, xaxis_title="log-moneyness k",
                      yaxis_title="total variance w",
                      title=f"{date} — fitted SVI slices: any crossing = calendar arbitrage")
    fig.show()

In [ ]:
if bench is not None:
    worst = bench["full"][0].sort("svi_min_g").head(1).to_dicts()[0]
    p = {c: worst[c] for c in ("a", "b", "rho", "m", "sigma")}
    kk = np.linspace(-0.5, 0.5, 400)
    dens, g = risk_neutral_density(kk, p)
    fig = make_subplots(rows=1, cols=3, subplot_titles=("w(k)", "g(k)", "implied density"))
    fig.add_trace(go.Scatter(x=kk, y=svi_raw(kk, **p)), 1, 1)
    fig.add_trace(go.Scatter(x=kk, y=g), 1, 2)
    fig.add_trace(go.Scatter(x=kk, y=np.where(g < 0, g, np.nan), line=dict(color="red", width=4)), 1, 2)
    fig.add_trace(go.Scatter(x=kk, y=dens), 1, 3)
    for c in (2, 3): fig.add_hline(y=0, line_dash="dot", row=1, col=c)
    fig.update_layout(width=1100, height=380, showlegend=False,
                      title=f"Worst real slice: {worst['date']}, τ={worst['tau']:.3f} "
                            f"(min g = {worst['svi_min_g']:.4f}) — the Vogt phenomenon in the data")
    fig.show()

In [ ]:
if bench is not None:
    sf, dfull = bench["full"]
    d_inv = dfull.filter(pl.col("theta_inversions") > 0)
    if d_inv.height:
        date = d_inv["date"][0]
        sl = sf.filter(pl.col("date") == date).sort("tau")
        taus = sl["tau"].to_numpy(); th_raw = sl["theta_raw"].to_numpy()
        th_iso = pava_increasing(th_raw)
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=taus, y=th_raw, mode="lines+markers", name="raw θ (from quotes)"))
        fig.add_trace(go.Scatter(x=taus, y=th_iso, mode="lines+markers", name="isotonic θ (PAVA)",
                                 line=dict(dash="dash")))
        bad = np.where(np.diff(th_raw) < -1e-10)[0] + 1
        fig.add_trace(go.Scatter(x=taus[bad], y=th_raw[bad], mode="markers", name="inversion",
                                 marker=dict(color="red", size=11, symbol="x")))
        fig.update_layout(width=PLOT_W, height=420, xaxis_title="τ (years)",
                          yaxis_title="ATM total variance θ",
                          title=f"{date} — raw θ term structure has inversions; PAVA repairs it")
        fig.show()

## 10. Against the industry benchmark: the OptionMetrics smoothed surface

`volatility_surface_clean.parquet` (NB01) is OptionMetrics' own proprietary smoothing on a standardized grid — the *industry* reference. For each day we evaluate our fitted SVI at the OM grid points (converting their `impl_strike` to log-moneyness with the forward interpolated from `forward_clean.parquet`) and measure the gap. A small gap says our transparent, replicable benchmark matches the commercial product. Skipped gracefully when the files are absent.

In [17]:
VS_PATH, FWD_PATH = OUT_DIR / "volatility_surface_clean.parquet", OUT_DIR / "forward_clean.parquet"
if bench is None or not (VS_PATH.exists() and FWD_PATH.exists()):
    print("[info] OM-surface comparison skipped (files not found - produced by NB01 on the real dataset).")
else:
    vs = pl.read_parquet(VS_PATH); fw = pl.read_parquet(FWD_PATH)
    fw = fw.with_columns((pl.col("expiration") - pl.col("date")).dt.total_days().alias("dte")) \
           .select(["date", "dte", "ForwardPrice"]).sort(["date", "dte"])
    slice_full, _ = bench["full"]
    gaps = []
    for d in slice_full["date"].unique().sort().to_list()[:min(LIMIT_DATES or 8, 8)]:
        day_slices = slice_full.filter(pl.col("date") == d)
        vsd = vs.filter((pl.col("date") == d) & (pl.col("days") >= 7)).sort("days")
        fwd_d = fw.filter(pl.col("date") == d)
        if vsd.height == 0 or fwd_d.height == 0:
            continue
        for row in vsd.iter_rows(named=True):
            tau_om = row["days"] / 365.0
            near = day_slices.with_columns((pl.col("tau") - tau_om).abs().alias("dt")).sort("dt").head(1)
            if near.height == 0 or float(near["dt"][0]) > 0.15 * tau_om + 1e-9:
                continue
            F = float(np.interp(row["days"], fwd_d["dte"].to_numpy(), fwd_d["ForwardPrice"].to_numpy()))
            k_om = float(np.log(row["impl_strike"] / F))
            p = {c: float(near[c][0]) for c in ("a", "b", "rho", "m", "sigma")}
            iv_ours = float(np.sqrt(max(svi_raw(k_om, **p), 1e-12) / float(near["tau"][0])))
            gaps.append(abs(iv_ours - row["impl_volatility"]))
    if gaps:
        g = np.array(gaps) * 100
        print(f"|our SVI - OptionMetrics surface| over {len(g)} grid points: "
              f"median {np.median(g):.3f} vol pts | 90th pct {np.quantile(g, .9):.3f}")
    else:
        print("[info] no comparable (date, maturity) pairs found.")


|our SVI - OptionMetrics surface| over 2958 grid points: median 0.931 vol pts | 90th pct 3.132


## 11. Save benchmark outputs (per domain)

In [18]:
if bench is not None:
    for name, (slice_df, day_df) in bench.items():
        slice_df.write_parquet(OUT_DIR / f"benchmark_svi_slices_{name}.parquet")
        day_df.write_parquet(OUT_DIR / f"benchmark_ssvi_days_{name}.parquet")
        print("written:", OUT_DIR / f"benchmark_svi_slices_{name}.parquet",
              "|", OUT_DIR / f"benchmark_ssvi_days_{name}.parquet")


written: data/clean/benchmark_svi_slices_full.parquet | data/clean/benchmark_ssvi_days_full.parquet
written: data/clean/benchmark_svi_slices_paper.parquet | data/clean/benchmark_ssvi_days_paper.parquet


## 12. Summary

**Replication (Sections 1–7, validated to the paper's digits):** three parameterizations and exact conversions (Vogt JW reproduced), butterfly/wing/crossedness diagnostics, quasi-explicit calibration, the paper's §5.2 recipe, §5.1 butterfly elimination (Example 5.1 exact), the three SSVI variants with Theorems 4.1/4.2, arbitrage-free interpolation/extrapolation.

**Benchmark v2 (Sections 8–11) — what changed and why it matters:**
- the **calendar check no longer sorts** the ATM variances: term-structure inversions are now *visible*, counted per day, and repaired by **isotonic regression** before the SSVI fit; $\theta$ is estimated from quotes, not SVI extrapolation;
- **hold-out scoring** (20% withheld per slice) puts SVI/SSVI on the same generalization footing as NB03/NB04;
- **liquidity weighting** stops wide-spread wing quotes from driving the fit;
- **cross-slice crossedness** completes the arbitrage audit (butterfly per slice + calendar between slices);
- the benchmark runs on **both domains** — the full quoted domain (honest difficulty; the bucket analysis shows arbitrage risk concentrated in 7–14d slices) and the paper-style restricted domain (literature-comparable numbers) — and both are saved for NB05;
- a direct gap measurement against the **OptionMetrics proprietary surface** anchors the benchmark to the industry reference.

**Outputs:** `benchmark_svi_slices_{full,paper}.parquet`, `benchmark_ssvi_days_{full,paper}.parquet`.
